In [1]:
import pandas as pd
import numpy as np
import re

print("Libraries loaded successfully.")


Libraries loaded successfully.


In [2]:
df = pd.read_csv("data/dss_raw.csv")

print("Rows:", len(df))
print("Columns:", len(df.columns))

df.head()

Rows: 2500
Columns: 15


,DOI,Title,Year,Date,Author,Institution,Country,Citation count,Open access,Abstract,Keyword,Topic,Subfield,Field,Domain
0,https://doi.org/10.1016/j.dss.2010.12.006,Cloud computing — The business perspective,2010,2010-12-25,Sean R. Marston|Zhi Li|Subhajyoti Bandyopadhya...,University of Florida|,US|US|US|US|,2360,NOT Open Access,NaN,Cloud computing|Perspective (graphical)|Comput...,Cloud Computing and Resource Management,Information Systems,Computer Science,Physical Sciences
1,https://doi.org/10.1016/j.dss.2012.06.008,The impact of electronic word-of-mouth communi...,2012,2012-07-10,Christy M.K. Cheung|Dimple R. Thadani,Hong Kong Baptist University|City University o...,HK|HK,1687,NOT Open Access,NaN,Scope (computer science)|Word of mouth|Key (lo...,Digital Marketing and Social Media,Sociology and Political Science,Social Sciences,Social Sciences
2,https://doi.org/10.1016/j.dss.2015.03.008,Recommender system application developments: A...,2015,2015-04-05,Jie Lü|Dianshuang Wu|Mingsong Mao|Wei Wang|Gua...,University of Technology Sydney|Centre for Qua...,AU|AU|AU|AU|AU,1587,NOT Open Access,NaN,Recommender system|Computer science|Informatio...,Recommender Systems and Techniques,Information Systems,Computer Science,Physical Sciences
3,https://doi.org/10.1016/j.dss.2010.08.006,The application of data mining techniques in f...,2010,2010-08-20,Eric W.T. Ngai|Yong Qiang Hu|Yung Hou Wong|Yij...,Hong Kong Polytechnic University|Guangdong Uni...,HK|CN|HK|CN|CN,1348,NOT Open Access,NaN,Credit card fraud|Computer science|Cluster ana...,Imbalanced Data Classification Techniques,Artificial Intelligence,Computer Science,Physical Sciences
4,https://doi.org/10.1016/j.dss.2010.02.008,Examining multi-dimensional trust and multi-fa...,2010,2010-02-26,Xin Robert Luo|Han Li|Jie Zhang|J. P. Shim,University of New Mexico|Minnesota State Unive...,US|US|US|US,1119,NOT Open Access,NaN,Risk perception|Artifact (error)|Perception|Te...,Technology Adoption and User Behaviour,Information Systems and Management,Decision Sciences,Social Sciences


In [3]:
df.columns.tolist()

['DOI',
 'Title',
 'Year',
 'Date',
 'Author',
 'Institution',
 'Country',
 'Citation count',
 'Open access',
 'Abstract',
 'Keyword',
 'Topic',
 'Subfield',
 'Field',
 'Domain']

In [4]:
summary = pd.DataFrame({
    "Data Type": df.dtypes,
    "Missing": df.isna().sum(),
    "Missing %": (df.isna().mean() * 100).round(2),
    "Unique Values": df.nunique()
})

summary

,Data Type,Missing,Missing %,Unique Values
DOI,object,0,0.00,2500
Title,object,0,0.00,2297
Year,int64,0,0.00,17
Date,object,0,0.00,1713
Author,object,214,8.56,2218
Institution,object,240,9.60,1896
Country,object,216,8.64,916
Citation count,int64,0,0.00,311
Open access,object,0,0.00,2
Abstract,object,2323,92.92,177


In [5]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 0


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2500 entries, 0 to 2499
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   DOI             2500 non-null   object
 1   Title           2500 non-null   object
 2   Year            2500 non-null   int64 
 3   Date            2500 non-null   object
 4   Author          2286 non-null   object
 5   Institution     2260 non-null   object
 6   Country         2284 non-null   object
 7   Citation count  2500 non-null   int64 
 8   Open access     2500 non-null   object
 9   Abstract        177 non-null    object
 10  Keyword         2490 non-null   object
 11  Topic           2480 non-null   object
 12  Subfield        2480 non-null   object
 13  Field           2480 non-null   object
 14  Domain          2480 non-null   object
dtypes: int64(2), object(13)
memory usage: 293.1+ KB


In [7]:
# ==========================================
# STEP 1: CHECK DUPLICATES
# ==========================================

print("Total rows:", len(df))

print(
    "Exact duplicate rows:",
    df.duplicated().sum()
)

print(
    "Duplicate DOIs:",
    df["DOI"].duplicated().sum()
)

print(
    "Duplicate titles:",
    df["Title"].duplicated().sum()
)

Total rows: 2500
Exact duplicate rows: 0
Duplicate DOIs: 0
Duplicate titles: 203


In [8]:
duplicate_titles = df[
    df["Title"].duplicated(keep=False)
].sort_values(
    by=["Title", "Year"]
)

duplicate_titles[
    [
        "DOI",
        "Title",
        "Year",
        "Date"
    ]
].head(30)

,DOI,Title,Year,Date
2268,https://doi.org/10.1016/s0167-9236(10)00099-0,Author Index,2010,2010-08-18
2342,https://doi.org/10.1016/s0167-9236(10)00024-2,Author Index,2010,2010-02-09
2238,https://doi.org/10.1016/s0167-9236(11)00148-5,Author Index,2011,2011-09-17
2262,https://doi.org/10.1016/s0167-9236(12)00028-0,Author Index,2012,2012-02-21
2306,https://doi.org/10.1016/s0167-9236(12)00223-0,Author Index,2012,2012-08-27
2234,https://doi.org/10.1016/s0167-9236(19)30117-4,Call for Papers,2019,2019-07-01
2239,https://doi.org/10.1016/s0167-9236(19)30155-1,Call for Papers,2019,2019-08-14
2253,https://doi.org/10.1016/s0167-9236(19)30136-8,Call for Papers,2019,2019-07-15
2260,https://doi.org/10.1016/s0167-9236(19)30180-0,Call for Papers,2019,2019-08-31
2263,https://doi.org/10.1016/j.dss.2019.04.003,Call for Papers,2019,2019-04-12


In [9]:
# ==========================================
# STEP 2: BASIC TEXT CLEANING
# ==========================================

import re


def clean_text(value):
    if pd.isna(value):
        return value

    value = str(value)

    # Remove line breaks and tabs
    value = value.replace("\n", " ")
    value = value.replace("\r", " ")
    value = value.replace("\t", " ")

    # Remove excessive spaces
    value = re.sub(r"\s+", " ", value)

    return value.strip()


# Clean article titles
df["Title"] = df["Title"].apply(clean_text)


# Clean DOI
df["DOI"] = (
    df["DOI"]
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(
        "https://doi.org/",
        "",
        regex=False
    )
)

In [11]:
# ==========================================
# STEP 3: CLEAN DATE
# ==========================================

df["Date"] = pd.to_datetime(
    df["Date"],
    errors="coerce"
)

print(df["Date"].dtype)

print(
    "Invalid dates:",
    df["Date"].isna().sum()
)

print(
    "Earliest date:",
    df["Date"].min()
)

print(
    "Latest date:",
    df["Date"].max()
)

datetime64[ns]
Invalid dates: 0
Earliest date: 2010-01-05 00:00:00
Latest date: 2026-09-09 00:00:00


In [12]:
year_mismatch = df[
    df["Year"] != df["Date"].dt.year
]

print(
    "Year/date mismatches:",
    len(year_mismatch)
)

year_mismatch[
    ["Title", "Year", "Date"]
].head()

Year/date mismatches: 0


,Title,Year,Date


In [13]:
# ==========================================
# STEP 4: CLEAN MULTI-VALUE COLUMNS
# ==========================================

def clean_multi_value(value):
    """
    Cleans values separated by |
    while keeping one article per row.
    """

    if pd.isna(value):
        return pd.NA

    # Split values
    values = str(value).split("|")

    cleaned = []
    seen = set()

    for item in values:

        # Clean spaces
        item = item.strip()

        # Ignore empty values
        if not item:
            continue

        # Remove duplicate values
        # Case-insensitive comparison
        key = item.lower()

        if key not in seen:
            cleaned.append(item)
            seen.add(key)

    if not cleaned:
        return pd.NA

    return " | ".join(cleaned)

In [14]:
multi_columns = [
    "Author",
    "Institution",
    "Country",
    "Keyword",
    "Topic",
    "Subfield",
    "Field",
    "Domain"
]

for column in multi_columns:
    df[column] = df[column].apply(clean_multi_value)

print("Multi-value columns cleaned.")

Multi-value columns cleaned.


In [15]:
df[
    [
        "Author",
        "Institution",
        "Country",
        "Keyword",
        "Topic"
    ]
].head(10)

,Author,Institution,Country,Keyword,Topic
0,Sean R. Marston | Zhi Li | Subhajyoti Bandyopa...,University of Florida,US,Cloud computing | Perspective (graphical) | Co...,Cloud Computing and Resource Management
1,Christy M.K. Cheung | Dimple R. Thadani,Hong Kong Baptist University | City University...,HK,Scope (computer science) | Word of mouth | Key...,Digital Marketing and Social Media
2,Jie Lü | Dianshuang Wu | Mingsong Mao | Wei Wa...,University of Technology Sydney | Centre for Q...,AU,Recommender system | Computer science | Inform...,Recommender Systems and Techniques
3,Eric W.T. Ngai | Yong Qiang Hu | Yung Hou Wong...,Hong Kong Polytechnic University | Guangdong U...,HK | CN,Credit card fraud | Computer science | Cluster...,Imbalanced Data Classification Techniques
4,Xin Robert Luo | Han Li | Jie Zhang | J. P. Shim,University of New Mexico | Minnesota State Uni...,US,Risk perception | Artifact (error) | Perceptio...,Technology Adoption and User Behaviour
5,Christy M.K. Cheung | Matthew Andrew Lee,Hong Kong Baptist University | City University...,HK,Reputation | Purchasing | Advertising | Word o...,Digital Marketing and Social Media
6,Stefan A. Seuring,University of Kassel,DE,Supply chain | Computer science | Sustainabili...,Sustainable Supply Chain Management
7,Siddhartha Bhattacharyya | Sanjeev Jha | Kuria...,University of Illinois Urbana-Champaign | Univ...,US,Credit card | Credit card fraud | Chargeback |...,Imbalanced Data Classification Techniques
8,Gaurav Kumar Bansal | Fatemeh Mariam Zahedi | ...,University of Wisconsin–Green Bay | University...,US,Personally identifiable information | Internet...,"Privacy, Security, and Data Protection"
9,Tao Zhou,Hangzhou Dianzi University,CN,Continuance | Mobile payment | Service quality...,Technology Adoption and User Behaviour


In [16]:
# ==========================================
# STEP 5: CREATE COUNT COLUMNS
# ==========================================

def count_multi_values(value):

    if pd.isna(value):
        return 0

    return len(
        [
            item
            for item in str(value).split("|")
            if item.strip()
        ]
    )


df["Author count"] = (
    df["Author"].apply(count_multi_values)
)

df["Institution count"] = (
    df["Institution"].apply(count_multi_values)
)

df["Country count"] = (
    df["Country"].apply(count_multi_values)
)

df["Keyword count"] = (
    df["Keyword"].apply(count_multi_values)
)

df["Topic count"] = (
    df["Topic"].apply(count_multi_values)
)

In [17]:
df[
    [
        "Title",
        "Author count",
        "Institution count",
        "Country count",
        "Keyword count",
        "Topic count"
    ]
].head(10)

,Title,Author count,Institution count,Country count,Keyword count,Topic count
0,Cloud computing — The business perspective,5,1,1,13,1
1,The impact of electronic word-of-mouth communi...,2,2,1,7,1
2,Recommender system application developments: A...,5,2,1,14,1
3,The application of data mining techniques in f...,5,2,2,12,1
4,Examining multi-dimensional trust and multi-fa...,4,4,1,17,1
5,What drives consumers to spread electronic wor...,2,2,1,18,1
6,A review of modeling approaches for sustainabl...,1,1,1,13,1
7,Data mining for credit card fraud: A comparati...,4,5,1,13,1
8,The impact of personal dispositions on informa...,3,3,1,16,1
9,An empirical examination of continuance intent...,1,1,1,19,1


In [18]:
# ==========================================
# STEP 6: COLLABORATION INDICATORS
# ==========================================

df["International collaboration"] = (
    df["Country count"] > 1
)

df["Multi-institution collaboration"] = (
    df["Institution count"] > 1
)

In [20]:
df.head()

,DOI,Title,Year,Date,Author,Institution,Country,Citation count,Open access,Abstract,...,Subfield,Field,Domain,Author count,Institution count,Country count,Keyword count,Topic count,International collaboration,Multi-institution collaboration
0,10.1016/j.dss.2010.12.006,Cloud computing — The business perspective,2010,2010-12-25,Sean R. Marston | Zhi Li | Subhajyoti Bandyopa...,University of Florida,US,2360,NOT Open Access,NaN,...,Information Systems,Computer Science,Physical Sciences,5,1,1,13,1,False,False
1,10.1016/j.dss.2012.06.008,The impact of electronic word-of-mouth communi...,2012,2012-07-10,Christy M.K. Cheung | Dimple R. Thadani,Hong Kong Baptist University | City University...,HK,1687,NOT Open Access,NaN,...,Sociology and Political Science,Social Sciences,Social Sciences,2,2,1,7,1,False,True
2,10.1016/j.dss.2015.03.008,Recommender system application developments: A...,2015,2015-04-05,Jie Lü | Dianshuang Wu | Mingsong Mao | Wei Wa...,University of Technology Sydney | Centre for Q...,AU,1587,NOT Open Access,NaN,...,Information Systems,Computer Science,Physical Sciences,5,2,1,14,1,False,True
3,10.1016/j.dss.2010.08.006,The application of data mining techniques in f...,2010,2010-08-20,Eric W.T. Ngai | Yong Qiang Hu | Yung Hou Wong...,Hong Kong Polytechnic University | Guangdong U...,HK | CN,1348,NOT Open Access,NaN,...,Artificial Intelligence,Computer Science,Physical Sciences,5,2,2,12,1,True,True
4,10.1016/j.dss.2010.02.008,Examining multi-dimensional trust and multi-fa...,2010,2010-02-26,Xin Robert Luo | Han Li | Jie Zhang | J. P. Shim,University of New Mexico | Minnesota State Uni...,US,1119,NOT Open Access,NaN,...,Information Systems and Management,Decision Sciences,Social Sciences,4,4,1,17,1,False,True


In [21]:
df["Open access"].value_counts(dropna=False)

Open access
NOT Open Access    1921
Open Access         579
Name: count, dtype: int64

In [22]:
# ==========================================
# STEP 7: CLEAN OPEN ACCESS
# ==========================================

def clean_open_access(value):

    if pd.isna(value):
        return "Unknown"

    value = str(value).strip().lower()

    if value in ["true", "yes", "1", "open", "open access"]:
        return "Open Access"

    if value in ["false", "no", "0", "closed", "closed access"]:
        return "Closed Access"

    return "Unknown"


df["Open access"] = (
    df["Open access"]
    .apply(clean_open_access)
)

df["Open access"].value_counts()

Open access
Unknown        1921
Open Access     579
Name: count, dtype: int64

In [23]:
# ==========================================
# STEP 8: ABSTRACT AVAILABILITY
# ==========================================

df["Has abstract"] = (
    df["Abstract"].notna()
)

print(
    df["Has abstract"].value_counts()
)

Has abstract
False    2323
True      177
Name: count, dtype: int64


In [24]:
# ==========================================
# STEP 9: DATA AVAILABILITY FLAGS
# ==========================================

df["Has author"] = df["Author"].notna()

df["Has institution"] = (
    df["Institution"].notna()
)

df["Has country"] = (
    df["Country"].notna()
)

df["Has keyword"] = (
    df["Keyword"].notna()
)

df["Has topic"] = (
    df["Topic"].notna()
)

In [25]:
availability = pd.DataFrame({
    "Available": [
        df["Has author"].sum(),
        df["Has institution"].sum(),
        df["Has country"].sum(),
        df["Has abstract"].sum(),
        df["Has keyword"].sum(),
        df["Has topic"].sum()
    ]
}, index=[
    "Author",
    "Institution",
    "Country",
    "Abstract",
    "Keyword",
    "Topic"
])

availability["Missing"] = (
    len(df) - availability["Available"]
)

availability["Coverage %"] = (
    availability["Available"]
    / len(df)
    * 100
).round(2)

availability

,Available,Missing,Coverage %
Author,2286,214,91.44
Institution,2260,240,90.40
Country,2260,240,90.40
Abstract,177,2323,7.08
Keyword,2490,10,99.60
Topic,2480,20,99.20


In [26]:
# ==========================================
# STEP 10: VALIDATE CITATIONS
# ==========================================

df["Citation count"] = pd.to_numeric(
    df["Citation count"],
    errors="coerce"
).fillna(0).astype(int)

print(
    "Minimum citations:",
    df["Citation count"].min()
)

print(
    "Maximum citations:",
    df["Citation count"].max()
)

print(
    "Total citations:",
    df["Citation count"].sum()
)

print(
    "Average citations:",
    round(df["Citation count"].mean(), 2)
)

Minimum citations: 0
Maximum citations: 2360
Total citations: 153626
Average citations: 61.45


In [27]:
# ==========================================
# STEP 11: FINAL VALIDATION
# ==========================================

print("=" * 50)
print("FINAL DSS DATASET")
print("=" * 50)

print("Rows:", len(df))
print("Columns:", len(df.columns))

print(
    "\nDuplicate DOIs:",
    df["DOI"].duplicated().sum()
)

print(
    "Missing DOI:",
    df["DOI"].isna().sum()
)

print(
    "Missing Title:",
    df["Title"].isna().sum()
)

print(
    "Missing Year:",
    df["Year"].isna().sum()
)

print(
    "Invalid Date:",
    df["Date"].isna().sum()
)

print(
    "\nYear range:",
    df["Year"].min(),
    "-",
    df["Year"].max()
)

FINAL DSS DATASET
Rows: 2500
Columns: 28

Duplicate DOIs: 0
Missing DOI: 0
Missing Title: 0
Missing Year: 0
Invalid Date: 0

Year range: 2010 - 2026


In [28]:
# ==========================================
# STEP 12: SAVE CLEANED DSS DATA
# ==========================================

output_file = "data/dss_cleaned.csv"

df.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

print("=" * 50)
print("CLEANING COMPLETE")
print("=" * 50)

print(f"File: {output_file}")
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")
print(f"Total citations: {df['Citation count'].sum():,}")

print("\nDSS dataset is ready for Streamlit.")

CLEANING COMPLETE
File: data/dss_cleaned.csv
Rows: 2,500
Columns: 28
Total citations: 153,626

DSS dataset is ready for Streamlit.
